# Лекция: Дисперсионный анализ (ANOVA / MANOVA) в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 11** (адаптация с языка R на Python)

## Краткая теория

**Дисперсионный анализ (ANOVA)** проверяет H0: средние значения исследуемой величины по уровням фактора(ов) равны (фактор не влияет).

- **Однофакторный:** одна независимая категориальная переменная
- **Двухфакторный:** два фактора + взаимодействие (`y ~ A * B`)
- **MANOVA:** несколько зависимых переменных одновременно

В Python:
- `statsmodels.formula.api.ols` + `anova_lm`
- `statsmodels.multivariate.manova.MANOVA`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.multivariate.manova import MANOVA

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")
print("Библиотеки загружены")


---
## 1. Пример: двухфакторный ANOVA — ToothGrowth

60 морских свинок:
- **len** — длина зубов (зависимая)
- **supp** — способ: OJ (сок) / VC (витамин C)
- **dose** — доза: 0.5, 1, 2 мг


In [ ]:
url = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/ToothGrowth.csv"
tg = pd.read_csv(url, index_col=0)
print(tg.head())
print("\nРазмерность:", tg.shape)
print("\nЧастоты (supp x dose):")
print(pd.crosstab(tg["supp"], tg["dose"]))


In [ ]:
agg = tg.groupby(["supp", "dose"])["len"].agg(["mean", "std", "count"]).round(2)
print(agg)


### Модель: `len ~ supp * dose`

В R: `aov(len ~ supp * dose)`  
`*` означает главные эффекты + взаимодействие.


In [ ]:
tg["dose"] = tg["dose"].astype("category")
tg["supp"] = tg["supp"].astype("category")

model = smf.ols("len ~ supp * dose", data=tg).fit()
print(anova_lm(model, typ=2))
print("\n--- summary ---")
print(model.summary().tables[1])


**Интерпретация (типичный результат):**
- p для `supp` < 0.05 → способ введения влияет на длину зубов
- p для `dose` < 0.05 → доза влияет
- p для `supp:dose` < 0.05 → есть взаимодействие факторов


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=tg, x="dose", y="len", hue="supp", ax=axes[0])
axes[0].set_title("Boxplot: len по dose и supp")

sns.pointplot(data=tg, x="dose", y="len", hue="supp",
              errorbar="se", ax=axes[1])
axes[1].set_title("Interaction plot (средние ± SE)")

plt.tight_layout()
plt.show()


---
## 2. Пример: MANOVA — злаки (UScereal)

Многомерный тест: различаются ли группы по **нескольким** зависимым переменным сразу.

В R: `manova(cbind(y1,y2,y3) ~ factor)`


In [ ]:
url_c = "https://vincentarelbundock.github.io/Rdatasets/csv/MASS/UScereal.csv"
cereal = pd.read_csv(url_c, index_col=0)
print(cereal.columns.tolist())
print(cereal.head())


In [ ]:
cols_y = ["calories", "fat", "sugars"]
df_m = cereal[["shelf"] + cols_y].dropna()
df_m["shelf"] = df_m["shelf"].astype("category")

print("Средние по shelf:")
print(df_m.groupby("shelf")[cols_y].mean().round(2))
print("\nКовариационная матрица:")
print(df_m[cols_y].cov().round(2))


In [ ]:
maov = MANOVA.from_formula("calories + fat + sugars ~ shelf", data=df_m)
print(maov.mv_test())


In [ ]:
print("=== Одномерные ANOVA ===")
for y in cols_y:
    m = smf.ols(f"{y} ~ shelf", data=df_m).fit()
    tab = anova_lm(m, typ=2)
    print(f"\nResponse: {y}")
    print(tab)


---
## 3. Задание 1: DA.csv (вес иглокожих)

Двухфакторный ANOVA для **weight**:
1. `weight ~ priming * view`
2. `weight ~ priming * region`
3. `weight ~ priming * conditions`

Подставьте свой файл:
```python
da = pd.read_csv("DA.csv")
```


In [ ]:
# Шаблон для DA.csv (раскомментируйте при наличии файла)
# da = pd.read_csv("DA.csv")
# for formula in [
#     "weight ~ C(priming) * C(view)",
#     "weight ~ C(priming) * C(region)",
#     "weight ~ C(priming) * C(conditions)",
# ]:
#     print("=" * 50)
#     print(formula)
#     m = smf.ols(formula, data=da).fit()
#     print(anova_lm(m, typ=2))

print("Загрузите DA.csv и раскомментируйте код выше.")
print("p < 0.05 → фактор (или взаимодействие) значимо влияет на weight.")


---
## 4. Задание 2: university2.csv (MANOVA)

Зависимость выбора университета от рейтингов Yslov_educ, Yslov_job, Yslov_science.

```python
uni = pd.read_csv("university2.csv")
maov = MANOVA.from_formula(
    "Yslov_educ + Yslov_job + Yslov_science ~ C(University)",
    data=uni
)
print(maov.mv_test())
```


In [ ]:
np.random.seed(42)
n = 90
demo = pd.DataFrame({
    "University": np.repeat(["A", "B", "C"], n // 3),
    "Yslov_educ": np.concatenate([
        np.random.normal(3, 1, n // 3),
        np.random.normal(4, 1, n // 3),
        np.random.normal(5, 1, n // 3),
    ]),
    "Yslov_job": np.concatenate([
        np.random.normal(2.5, 1, n // 3),
        np.random.normal(3.5, 1, n // 3),
        np.random.normal(4.5, 1, n // 3),
    ]),
    "Yslov_science": np.concatenate([
        np.random.normal(3, 1.2, n // 3),
        np.random.normal(3.2, 1.2, n // 3),
        np.random.normal(4, 1.2, n // 3),
    ]),
})

print(demo.groupby("University")[["Yslov_educ", "Yslov_job", "Yslov_science"]].mean().round(2))

maov_demo = MANOVA.from_formula(
    "Yslov_educ + Yslov_job + Yslov_science ~ C(University)", data=demo
)
print(maov_demo.mv_test())


In [ ]:
print("=== Одномерные ANOVA (demo) ===")
for y in ["Yslov_educ", "Yslov_job", "Yslov_science"]:
    m = smf.ols(f"{y} ~ C(University)", data=demo).fit()
    print(f"\n{y}:")
    print(anova_lm(m, typ=2))


### Как описать результаты

1. **ANOVA:** для каждого эффекта (A, B, A:B) — F, p. Если p < 0.05 — эффект значим.
2. **Взаимодействие значимо** — влияние одного фактора зависит от уровня другого.
3. **MANOVA:** если p < 0.05, группы различаются в многомерном пространстве откликов.
4. После значимого MANOVA — одномерные ANOVA по каждой y.

---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `aov(y ~ A * B)` | `smf.ols("y ~ A * B", data=df).fit()` |
| `summary(aov)` | `anova_lm(model, typ=2)` |
| `aggregate(...)` | `df.groupby(["A","B"])["y"].mean()` |
| `manova(cbind(y1,y2) ~ F)` | `MANOVA.from_formula("y1 + y2 ~ F", data=df)` |
| `summary(manova)` | `maov.mv_test()` |
| interaction plot | `sns.pointplot(..., hue=...)` |

---
## Рекомендации

1. Факторы: `df["A"] = df["A"].astype("category")`.
2. Файлы **DA.csv** и **university2.csv** подставьте локально.
3. `pip install statsmodels seaborn`

**Удачи с выполнением Задания 11!**
